# LOGAN: Foldseek evidence for missed-gene candidates and matched controls

**Draft caption.** Structural (3Di+AA, solid) and sequence-only (AA, hatched) Foldseek evidence for 20,000 high-scoring intergenic ORFs ≥100 aa that Prodigal did not call (candidates, score ≥ the 0.90-precision threshold), and arms matched to them by nearest length and truncation: complete Prodigal genes (positive control), low-scoring intergenic ORFs, antisense and frameshift shadows, and the candidates with their 3Di strings shuffled (amino acids unchanged). (a) Hits with full-length mutual coverage (query and target coverage ≥0.8), the primary readout; (b) any hit at E ≤ 10⁻³. Databases: PDB100, AFDB/Swiss-Prot, CATH50, BFVD and AFDB50. The shuffled null has one full-length hit in 20,000.

**Outputs:** `../images/figL06_missed_gene_evidence.pdf` (vector) and `../images/figL06_missed_gene_evidence.png` (300 dpi).

**Data** (plotting-ready, committed; produced by `../prep/logan_prepare_data.py` on Setonix):

- `../data/figL06_evidence_rates.tsv.gz` (90 rows; sha256 `d9c1154ad24f…`): hit and full-length (qcov,tcov>=0.8) rates by arm, mode and database (five DBs). Source on Setonix (`$LOGAN_ROOT`): `qc/candidates/evidence_rates.tsv`; `qc/candidates_extra/evidence_rates.tsv`.

**Upstream.** LOGAN v1.2 metagenomes (50,009), genome_entropy #102 (closed by PR #115); analysis scripts `Validation/logan/scripts/` at commit `06080b4`. Full run provenance: `../data/logan_provenance.json`.

**Run:** `jupyter nbconvert --to notebook --execute figL06_missed_gene_evidence.ipynb` from `manuscript/notebooks/`. Needs only pandas, numpy, matplotlib and seaborn; no HPC or remote paths.

In [ ]:
from pathlib import Path

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

DATA, IMAGES = Path("../data"), Path("../images")
IMAGES.mkdir(exist_ok=True)
plt.rcParams["svg.hashsalt"] = "genome_entropy"   # deterministic SVG ids if SVG is ever written

# Colour slots (fixed order, colour-blind-checked categorical palette used across the LOGAN figures)
SLOTS = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#008300", "#4a3aa7"]
INK, INK2, GRID, SURFACE = "#0b0b0b", "#52514e", "#e4e3df", "#fcfcfb"
sns.set_theme(context="paper", style="ticks", font_scale=1.15, rc={
    "axes.edgecolor": INK2, "axes.labelcolor": INK, "xtick.color": INK2, "ytick.color": INK2,
    "text.color": INK, "axes.facecolor": SURFACE, "figure.facecolor": "white", "grid.color": GRID,
    "axes.grid": True, "grid.linewidth": 0.6, "axes.spines.top": False, "axes.spines.right": False,
    "legend.frameon": False, "lines.linewidth": 2, "font.family": "DejaVu Sans"})


def panel(ax, letter, x=-0.08, y=1.04):
    ax.text(x, y, letter, transform=ax.transAxes, fontsize=15, fontweight="bold", va="bottom", ha="right")


def save(fig, name):
    """Vector PDF (fixed metadata, so reruns are identical) and a 300-dpi PNG preview."""
    fig.savefig(IMAGES / f"{name}.pdf", bbox_inches="tight", metadata={"CreationDate": None, "ModDate": None})
    fig.savefig(IMAGES / f"{name}.png", dpi=300, bbox_inches="tight", metadata={"Software": None})
    plt.close(fig)
    print("wrote", IMAGES / f"{name}.pdf", IMAGES / f"{name}.png")

In [ ]:
R = pd.read_csv(DATA / "figL06_evidence_rates.tsv.gz", sep="\t")
R = R[R.db == "any"]
ARMS = ["prodigal", "candidate", "candidate_shuffled_3di", "intergenic_lo", "antisense", "frameshift"]
LABEL = {"prodigal": "complete Prodigal genes", "candidate": "high-score intergenic (candidates)",
         "candidate_shuffled_3di": "candidates, shuffled 3Di", "intergenic_lo": "low-score intergenic",
         "antisense": "antisense shadows", "frameshift": "frameshift shadows"}
COLOR = dict(zip(ARMS, [SLOTS[0], SLOTS[1], SLOTS[6], SLOTS[2], SLOTS[3], SLOTS[4]]))


def pct(v):
    return f"{100 * v:.3f}%" if v < 0.001 else f"{100 * v:.2f}%" if v < 0.01 else f"{100 * v:.1f}%"


fig, axes = plt.subplots(1, 2, figsize=(14, 5.2), sharey=True)
for ax, (metric, title), letter in zip(axes, (("fullcov_rate", "full-length hit (qcov and tcov ≥ 0.8)"),
                                              ("hit_rate", "any hit (E ≤ 10⁻³)")), "ab"):
    y = np.arange(len(ARMS))
    for off, mode, hatch in ((-0.19, "struct", None), (0.19, "seq", "///")):
        sel = R[R["mode"] == mode].set_index("arm")
        v = [sel.loc[a, metric] if a in sel.index else np.nan for a in ARMS]
        ax.barh(y + off, v, height=0.36, color=[COLOR[a] for a in ARMS], hatch=hatch, edgecolor="white", lw=0.5)
        for yi, vi in zip(y, v):
            if not np.isnan(vi):
                ax.text(vi + 0.008, yi + off, pct(vi), va="center", fontsize=8.5, color=INK2)
    ax.set_yticks(y, [LABEL[a] for a in ARMS])
    ax.invert_yaxis()
    ax.set_xlim(0, 1.05)
    ax.set_xlabel("fraction of queries")
    ax.set_title(title, loc="left", fontsize=11)
    panel(ax, letter, x=-0.02 if letter == "b" else -0.52)
axes[0].legend(handles=[plt.Rectangle((0, 0), 1, 1, fc=INK2), plt.Rectangle((0, 0), 1, 1, fc=INK2, hatch="///", ec="white")],
               labels=["3Di+AA (structure)", "AA only (sequence)"], loc="upper right")
fig.tight_layout()
save(fig, "figL06_missed_gene_evidence")